<a href="https://colab.research.google.com/github/alexandrumoldovan1/housing-prices-ml/blob/main/notebooks/04_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install ML libraries
!pip install xgboost lightgbm catboost optuna -q

# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import joblib
warnings.filterwarnings('ignore')

# Sklearn imports
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

# Boosting libraries
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Visualization
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.5 MB/s eta 0:00:00
Libraries imported successfully!


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define paths
DRIVE_PATH = '/content/drive/MyDrive/ColabProjects/housing-prices-ml'
PROCESSED_DATA_PATH = f'{DRIVE_PATH}/processed_data'
MODELS_PATH = f'{DRIVE_PATH}/models'
OUTPUTS_PATH = f'{DRIVE_PATH}/outputs'

# Load processed v2 datasets (with PLUTO + distance features)
X_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_train_v2.parquet')
X_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_test_v2.parquet')
X_train_scaled = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_train_scaled_v2.parquet')
X_test_scaled = pd.read_parquet(f'{PROCESSED_DATA_PATH}/X_test_scaled_v2.parquet')
y_train = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_train_v2.parquet').squeeze()
y_test = pd.read_parquet(f'{PROCESSED_DATA_PATH}/y_test_v2.parquet').squeeze()

print(f"Data v2 loaded successfully!\n")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")
print(f"\nFeatures: {X_train.shape[1]} (v1 had 30, v2 has new geospatial features)")

Mounted at /content/drive
Data v2 loaded successfully!

X_train: (99880, 43)
X_test:  (53522, 43)
y_train: (99880,)
y_test:  (53522,)

Features: 43 (v1 had 30, v2 has new geospatial features)


In [3]:
# Define a reusable evaluation function for all models
def evaluate_model(model, X_tr, y_tr, X_te, y_te, model_name, use_cv=True, cv_folds=5):
    """
    Trains a model, evaluates on test set, and runs K-Fold Cross-Validation.
    Returns a dictionary with all metrics.
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")

    start_time = time.time()

    # Train on full training set
    model.fit(X_tr, y_tr)

    # Predict on test set (in log scale)
    y_pred_log = model.predict(X_te)

    # Convert predictions back to original $ scale
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_te)

    # Test set metrics (on original $ scale)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_te, y_pred_log)  # R² on log scale (more stable)

    # K-Fold Cross-Validation on training set (R² scoring)
    cv_score = None
    if use_cv:
        kf = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        cv_scores = cross_val_score(model, X_tr, y_tr, cv=kf,
                                     scoring='r2', n_jobs=-1)
        cv_score = cv_scores.mean()

    elapsed = time.time() - start_time

    # Print results
    print(f"   RMSE (test):  ${rmse:,.0f}")
    print(f"   MAE  (test):  ${mae:,.0f}")
    print(f"   R²   (test):  {r2:.4f}")
    if cv_score is not None:
        print(f"   R²   (CV-{cv_folds}): {cv_score:.4f}")
    print(f"   Training time: {elapsed:.1f}s")

    return {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'R2_test': r2,
        'R2_cv': cv_score,
        'Time_sec': elapsed,
        'trained_model': model
    }

# Initialize results storage
all_results = []

print("Evaluation function defined.")
print("\nMetrics explanation:")
print("   RMSE: Root Mean Squared Error (in $)")
print("   MAE:  Mean Absolute Error (in $)")
print("   R²:   Coefficient of determination (0-1, higher=better)")
print("   CV:   K-Fold Cross-Validation score")

Evaluation function defined.

Metrics explanation:
   RMSE: Root Mean Squared Error (in $)
   MAE:  Mean Absolute Error (in $)
   R²:   Coefficient of determination (0-1, higher=better)
   CV:   K-Fold Cross-Validation score


In [4]:
# Reload already-trained models from previous session
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def quick_evaluate(model, X_te, y_te, model_name, train_time=0):
    """Evaluate without retraining (model already trained)"""
    y_pred_log = model.predict(X_te)
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_te)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_te, y_pred_log)

    print(f"{model_name}: R² = {r2:.4f}, RMSE = ${rmse:,.0f}, MAE = ${mae:,.0f}")
    return {'Model': model_name, 'RMSE': rmse, 'MAE': mae, 'R2_test': r2,
            'R2_cv': None, 'Time_sec': train_time, 'trained_model': model}

# Load saved models from yesterday
print("Reloading models from previous session...\n")

for filename, name, time_sec in [
    ('linear_regression.pkl', 'Linear Regression', 6.3),
    ('ridge.pkl', 'Ridge Regression', 0.5),
    ('lasso.pkl', 'Lasso Regression', 67.0),
]:
    model = joblib.load(f'{MODELS_PATH}/{filename}')
    result = quick_evaluate(model, X_test_scaled, y_test, name, time_sec)
    all_results.append(result)

print(f"\n{len(all_results)} models reloaded successfully.")

Reloading models from previous session...

Linear Regression: R² = 0.3407, RMSE = $13,779,156, MAE = $2,228,055
Ridge Regression: R² = 0.3410, RMSE = $13,777,099, MAE = $2,227,505
Lasso Regression: R² = 0.3416, RMSE = $13,773,645, MAE = $2,226,181

3 models reloaded successfully.


In [ ]:
# Model 1: Linear Regression (baseline)
model = LinearRegression()
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "Linear Regression")
all_results.append(result)

# Save the trained model
joblib.dump(result['trained_model'], f'{MODELS_PATH}/linear_regression.pkl')
print(f"\nModel saved to: {MODELS_PATH}/linear_regression.pkl")


Training: Linear Regression
   RMSE (test):  $13,779,156
   MAE  (test):  $2,228,055
   R²   (test):  0.3407
   R²   (CV-5): 0.4241
   Training time: 6.3s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/linear_regression.pkl


In [ ]:
# Model 2: Ridge Regression
def objective_ridge(trial):
    alpha = trial.suggest_float('alpha', 0.01, 100, log=True)
    model = Ridge(alpha=alpha, random_state=42)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning Ridge hyperparameters with Optuna (20 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_ridge, n_trials=20, show_progress_bar=False)
print(f"Best alpha: {study.best_params['alpha']:.4f}")

model = Ridge(alpha=study.best_params['alpha'], random_state=42)
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "Ridge Regression")
all_results.append(result)

joblib.dump(result['trained_model'], f'{MODELS_PATH}/ridge.pkl')
print(f"\nModel saved to: {MODELS_PATH}/ridge.pkl")

Tuning Ridge hyperparameters with Optuna (20 trials)...
Best alpha: 0.7179

Training: Ridge Regression
   RMSE (test):  $13,777,099
   MAE  (test):  $2,227,505
   R²   (test):  0.3410
   R²   (CV-5): 0.4241
   Training time: 0.5s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/ridge.pkl


In [ ]:
# Model 3: Lasso Regression
def objective_lasso(trial):
    alpha = trial.suggest_float('alpha', 0.0001, 10, log=True)
    model = Lasso(alpha=alpha, random_state=42, max_iter=5000)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning Lasso hyperparameters with Optuna (20 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_lasso, n_trials=20, show_progress_bar=False)
print(f"Best alpha: {study.best_params['alpha']:.6f}")

model = Lasso(alpha=study.best_params['alpha'], random_state=42, max_iter=5000)
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "Lasso Regression")
all_results.append(result)

non_zero_features = np.sum(result['trained_model'].coef_ != 0)
total_features = len(result['trained_model'].coef_)
print(f"\nLasso selected {non_zero_features}/{total_features} features")

joblib.dump(result['trained_model'], f'{MODELS_PATH}/lasso.pkl')
print(f"Model saved to: {MODELS_PATH}/lasso.pkl")

Tuning Lasso hyperparameters with Optuna (20 trials)...
Best alpha: 0.000424

Training: Lasso Regression
   RMSE (test):  $13,773,645
   MAE  (test):  $2,226,181
   R²   (test):  0.3416
   R²   (CV-5): 0.4238
   Training time: 67.0s

Lasso selected 37/43 features
Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/lasso.pkl


In [5]:
# Model 4: KNN Regressor
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 5, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    p = trial.suggest_int('p', 1, 2)

    model = KNeighborsRegressor(n_neighbors=n_neighbors, weights=weights,
                                  p=p, n_jobs=-1)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning KNN hyperparameters with Optuna (15 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_knn, n_trials=15, show_progress_bar=False)
print(f"Best params: {study.best_params}")

model = KNeighborsRegressor(**study.best_params, n_jobs=-1)
result = evaluate_model(model, X_train_scaled, y_train, X_test_scaled, y_test,
                         "KNN Regressor")
all_results.append(result)

joblib.dump(result['trained_model'], f'{MODELS_PATH}/knn.pkl')
print(f"Model saved to: {MODELS_PATH}/knn.pkl")

Tuning KNN hyperparameters with Optuna (15 trials)...
Best params: {'n_neighbors': 12, 'weights': 'distance', 'p': 1}

Training: KNN Regressor
   RMSE (test):  $13,780,219
   MAE  (test):  $2,151,712
   R²   (test):  0.3231
   R²   (CV-5): 0.6333
   Training time: 434.9s
Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/knn.pkl


In [6]:
# Model 5: SVR (optimized for speed)
print("Preparing sample for SVR...")
sample_size = 15000
np.random.seed(42)
sample_idx = np.random.choice(X_train_scaled.index, size=sample_size, replace=False)
X_train_svr = X_train_scaled.loc[sample_idx]
y_train_svr = y_train.loc[sample_idx]

print(f"   Sample size: {len(X_train_svr):,} rows")

def objective_svr(trial):
    C = trial.suggest_float('C', 0.1, 10, log=True)
    epsilon = trial.suggest_float('epsilon', 0.01, 1.0, log=True)
    model = SVR(kernel='rbf', C=C, epsilon=epsilon, gamma='scale')
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_svr, y_train_svr, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("\nTuning SVR hyperparameters with Optuna (6 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_svr, n_trials=6, show_progress_bar=False)
print(f"Best params: {study.best_params}")

model = SVR(kernel='rbf', **study.best_params, gamma='scale')
print("\nTraining final SVR model...")
start_time = time.time()
model.fit(X_train_svr, y_train_svr)
print(f"   Trained in {time.time() - start_time:.1f}s")

result = evaluate_model(model, X_train_svr, y_train_svr, X_test_scaled, y_test,
                         "SVR (sample 15K)", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/svr.pkl')
print(f"\nModel saved to: {MODELS_PATH}/svr.pkl")

Preparing sample for SVR...
   Sample size: 15,000 rows

Tuning SVR hyperparameters with Optuna (6 trials)...
Best params: {'C': 4.732804251815262, 'epsilon': 0.2488182340338328}

Training final SVR model...
   Trained in 11.4s

Training: SVR (sample 15K)
   RMSE (test):  $13,679,562
   MAE  (test):  $2,177,968
   R²   (test):  0.3569
   Training time: 40.1s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/svr.pkl


In [7]:
# Model 6: Random Forest Regressor
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    max_depth = trial.suggest_int('max_depth', 10, 25)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = RandomForestRegressor(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
        n_jobs=-1, random_state=42
    )
    kf = KFold(n_splits=2, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning Random Forest hyperparameters with Optuna (7 trials)...")
print("Estimated time: 30-50 minutes...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_rf, n_trials=7, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final Random Forest model...")
model = RandomForestRegressor(**study.best_params, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "Random Forest", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/random_forest.pkl')
print(f"\nModel saved to: {MODELS_PATH}/random_forest.pkl")

Tuning Random Forest hyperparameters with Optuna (7 trials)...
Estimated time: 30-50 minutes...
Best params: {'n_estimators': 153, 'max_depth': 17, 'min_samples_split': 2, 'min_samples_leaf': 1}

Training final Random Forest model...

Training: Random Forest
   RMSE (test):  $13,693,031
   MAE  (test):  $2,060,877
   R²   (test):  0.4029
   Training time: 200.5s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/random_forest.pkl


In [8]:
# Model 7: XGBoost
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = XGBRegressor(**params, n_jobs=-1, random_state=42, verbosity=0)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning XGBoost hyperparameters with Optuna (10 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_xgb, n_trials=10, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final XGBoost model...")
model = XGBRegressor(**study.best_params, n_jobs=-1, random_state=42, verbosity=0)
result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "XGBoost", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/xgboost.pkl')
print(f"\nModel saved to: {MODELS_PATH}/xgboost.pkl")

Tuning XGBoost hyperparameters with Optuna (10 trials)...
Best params: {'n_estimators': 352, 'max_depth': 9, 'learning_rate': 0.04036115763468125, 'subsample': 0.8529075233281387, 'colsample_bytree': 0.6591503771658804, 'min_child_weight': 1}

Training final XGBoost model...

Training: XGBoost
   RMSE (test):  $13,685,450
   MAE  (test):  $2,041,047
   R²   (test):  0.4062
   Training time: 12.9s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/xgboost.pkl


In [9]:
# Model 8: LightGBM
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    }
    model = LGBMRegressor(**params, n_jobs=-1, random_state=42, verbose=-1)
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning LightGBM hyperparameters with Optuna (10 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_lgbm, n_trials=10, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final LightGBM model...")
model = LGBMRegressor(**study.best_params, n_jobs=-1, random_state=42, verbose=-1)
result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "LightGBM", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/lightgbm.pkl')
print(f"\nModel saved to: {MODELS_PATH}/lightgbm.pkl")

Tuning LightGBM hyperparameters with Optuna (10 trials)...
Best params: {'n_estimators': 464, 'max_depth': 15, 'learning_rate': 0.12543935841074233, 'num_leaves': 87, 'subsample': 0.8939434377768167, 'colsample_bytree': 0.6372346007955749}

Training final LightGBM model...

Training: LightGBM
   RMSE (test):  $13,649,896
   MAE  (test):  $2,041,339
   R²   (test):  0.4132
   Training time: 7.0s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/lightgbm.pkl


In [10]:
# Model 9: CatBoost
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 500),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    }
    model = CatBoostRegressor(**params, random_state=42, verbose=0,
                                 thread_count=-1, bootstrap_type='Bernoulli')
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                              scoring='r2', n_jobs=-1)
    return scores.mean()

print("Tuning CatBoost hyperparameters with Optuna (8 trials)...")
print("This may take 10-15 minutes...")
study = optuna.create_study(direction='maximize')
study.optimize(objective_cat, n_trials=8, show_progress_bar=False)
print(f"Best params: {study.best_params}")

print("\nTraining final CatBoost model...")
model = CatBoostRegressor(**study.best_params, random_state=42, verbose=0,
                            thread_count=-1, bootstrap_type='Bernoulli')
result = evaluate_model(model, X_train, y_train, X_test, y_test,
                         "CatBoost", use_cv=False)
all_results.append(result)

joblib.dump(model, f'{MODELS_PATH}/catboost.pkl')
print(f"\nModel saved to: {MODELS_PATH}/catboost.pkl")

Tuning CatBoost hyperparameters with Optuna (8 trials)...
This may take 10-15 minutes...
Best params: {'iterations': 498, 'depth': 10, 'learning_rate': 0.09414985790224219, 'l2_leaf_reg': 9.671490769036641, 'subsample': 0.6000955134062992}

Training final CatBoost model...

Training: CatBoost
   RMSE (test):  $13,705,054
   MAE  (test):  $2,059,405
   R²   (test):  0.3941
   Training time: 49.0s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/catboost.pkl


In [11]:
# Model 10: Stacking Regressor (combines top 3 boosting models)
print("Building Stacking Regressor...")
print("Base learners: XGBoost, LightGBM, CatBoost")
print("Meta-learner: Ridge Regression\n")

# Load the best trained models directly (no need to recreate)
xgb_model = joblib.load(f'{MODELS_PATH}/xgboost.pkl')
lgbm_model = joblib.load(f'{MODELS_PATH}/lightgbm.pkl')
cat_model = joblib.load(f'{MODELS_PATH}/catboost.pkl')

base_models = [
    ('xgb', xgb_model),
    ('lgbm', lgbm_model),
    ('cat', cat_model)
]

meta_model = Ridge(alpha=1.0)

stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_model,
    cv=3,
    n_jobs=-1
)

result = evaluate_model(stacking_model, X_train, y_train, X_test, y_test,
                         "Stacking Regressor", use_cv=False)
all_results.append(result)

joblib.dump(result['trained_model'], f'{MODELS_PATH}/stacking.pkl')
print(f"\nModel saved to: {MODELS_PATH}/stacking.pkl")

Building Stacking Regressor...
Base learners: XGBoost, LightGBM, CatBoost
Meta-learner: Ridge Regression


Training: Stacking Regressor
   RMSE (test):  $13,676,190
   MAE  (test):  $2,031,763
   R²   (test):  0.4196
   Training time: 242.2s

Model saved to: /content/drive/MyDrive/ColabProjects/housing-prices-ml/models/stacking.pkl


In [12]:
# Final comparison v1 vs v2
print("="*80)
print("NOTEBOOK 04 v2 - MODEL TRAINING SUMMARY (v2 with PLUTO + leakage fix)")
print("="*80)

# Build results DataFrame
results_df = pd.DataFrame(all_results)
results_df = results_df[['Model', 'R2_test', 'RMSE', 'MAE', 'Time_sec']]
results_df.columns = ['Model', 'R² (test)', 'RMSE ($)', 'MAE ($)', 'Time (s)']
results_df = results_df.sort_values('R² (test)', ascending=False).reset_index(drop=True)

# v1 results (from previous notebook)
v1_results = {
    'Linear Regression': 0.3530,
    'Ridge Regression': 0.3530,
    'Lasso Regression': 0.3530,
    'KNN Regressor': 0.3770,
    'SVR (sample 15K)': 0.2664,
    'Random Forest': 0.4057,
    'XGBoost': 0.4398,
    'LightGBM': 0.4292,
    'CatBoost': 0.4345,
    'Stacking Regressor': 0.4485,
}

results_df['R² v1'] = results_df['Model'].map(v1_results)
results_df['Δ (v2-v1)'] = (results_df['R² (test)'] - results_df['R² v1']).round(4)

# Display
display_df = results_df.copy()
display_df['R² (test)'] = display_df['R² (test)'].round(4)
display_df['R² v1'] = display_df['R² v1'].round(4)
display_df['RMSE ($)'] = display_df['RMSE ($)'].apply(lambda x: f'${x:,.0f}')
display_df['MAE ($)'] = display_df['MAE ($)'].apply(lambda x: f'${x:,.0f}')
display_df['Time (s)'] = display_df['Time (s)'].round(1)

print(f"\nAll 10 models ranked by R² (test):\n")
print(display_df[['Model', 'R² (test)', 'R² v1', 'Δ (v2-v1)', 'RMSE ($)', 'MAE ($)']].to_string(index=False))

# Key insight
print(f"\n{'='*80}")
print("KEY INSIGHTS:")
print(f"{'='*80}")
print(f"   Best model v2: Stacking Regressor (R² = {results_df.iloc[0]['R² (test)']:.4f})")
print(f"   Best model v1: Stacking Regressor (R² = 0.4485)")
print(f"   Net change:    -0.029")
print(f"\n   Interpretation:")
print(f"   - v1 contained 3 sources of data leakage (target encoding, price_per_sqft, agg stats)")
print(f"   - v2 corrected leakage AND added 18 geospatial features")
print(f"   - The 0.029 reduction represents 'leakage premium' (artificial R² gain)")
print(f"   - SVR improved by +0.090 with new features, validating geospatial signal")

# Save results
results_for_save = []
for r in all_results:
    r_copy = {k: v for k, v in r.items() if k != 'trained_model'}
    results_for_save.append(r_copy)

pd.DataFrame(results_for_save).to_csv(f'{OUTPUTS_PATH}/model_results_v2.csv', index=False)

print(f"\n\nResults saved to: {OUTPUTS_PATH}/model_results_v2.csv")
print("\n" + "="*80)
print("Notebook 04 v2 completed.")
print("Next step: Notebook 05 - Evaluation & Web App")
print("="*80)

NOTEBOOK 04 v2 - MODEL TRAINING SUMMARY (v2 with PLUTO + leakage fix)

All 10 models ranked by R² (test):

             Model  R² (test)  R² v1  Δ (v2-v1)    RMSE ($)    MAE ($)
Stacking Regressor     0.4196 0.4485    -0.0289 $13,676,190 $2,031,763
          LightGBM     0.4132 0.4292    -0.0160 $13,649,896 $2,041,339
           XGBoost     0.4062 0.4398    -0.0336 $13,685,450 $2,041,047
     Random Forest     0.4029 0.4057    -0.0028 $13,693,031 $2,060,877
          CatBoost     0.3941 0.4345    -0.0404 $13,705,054 $2,059,405
  SVR (sample 15K)     0.3569 0.2664     0.0905 $13,679,562 $2,177,968
  Lasso Regression     0.3416 0.3530    -0.0114 $13,773,645 $2,226,181
  Ridge Regression     0.3410 0.3530    -0.0120 $13,777,099 $2,227,505
 Linear Regression     0.3407 0.3530    -0.0123 $13,779,156 $2,228,055
     KNN Regressor     0.3231 0.3770    -0.0539 $13,780,219 $2,151,712

KEY INSIGHTS:
   Best model v2: Stacking Regressor (R² = 0.4196)
   Best model v1: Stacking Regressor (R² = 0.4